# 📈 Stock Trading Habit Tracker


## Step 1: Imports

In [2]:
import json
import os
from datetime import datetime, timedelta

## Step 2: The `Habit` class
Stores a habit's info and its completion history, and can calculate streaks.

In [3]:
class Habit:
    def __init__(self, name: str, description: str, periodicity: str):
        self.name = name
        self.description = description
        self.periodicity = periodicity
        self.created_date = datetime.now()
        self.completions = []

    def check_off(self):
        """Record a completion for right now."""
        now = datetime.now()
        self.completions.append(now)
        print(f"✓ Habit '{self.name}' checked off at {now.strftime('%Y-%m-%d %H:%M')}.")

    def edit(self, name=None, description=None, periodicity=None):
        """Update one or more fields. Leave an argument as None to keep it unchanged."""
        if name is not None:
            self.name = name
        if description is not None:
            self.description = description
        if periodicity is not None:
            self.periodicity = periodicity

    def get_streak(self): 
        if not self.completions:
            return 0
        sorted_dates = sorted(self.completions, reverse=True)
        if self.periodicity == "daily":
            return self._calculate_daily_streak(sorted_dates)
        else:
            return self._calculate_weekly_streak(sorted_dates)

    def _calculate_daily_streak(self, sorted_dates: list):
        today = datetime.now().date()
        streak = 0
        check_date = today
        completions_set = set(d.date() for d in sorted_dates)
        while check_date in completions_set:
            streak += 1
            check_date -= timedelta(days=1)
        return streak

    def _calculate_weekly_streak(self, sorted_dates: list):
        today = datetime.now().date()
        current_week_start = today - timedelta(days=today.weekday())
        completions_weeks = set(
            d.date() - timedelta(days=d.date().weekday()) for d in sorted_dates
        )
        streak = 0
        check_week = current_week_start
        while check_week in completions_weeks:
            streak += 1
            check_week -= timedelta(days=7)
        return streak

    def get_longest_streak(self):
        if not self.completions:
            return 0
        sorted_dates = sorted(self.completions)
        if self.periodicity == "daily":
            return self._longest_daily_streak(sorted_dates)
        else:
            return self._longest_weekly_streak(sorted_dates)

    def _longest_daily_streak(self, sorted_dates: list):
        completion_days = sorted(set(d.date() for d in sorted_dates))
        if not completion_days:
            return 0
        longest = 1
        current = 1
        for i in range(1, len(completion_days)):
            if (completion_days[i] - completion_days[i - 1]).days == 1:
                current += 1
                longest = max(longest, current)
            else:
                current = 1
        return longest

    def _longest_weekly_streak(self, sorted_dates: list):
        completion_weeks = sorted(set(
            d.date() - timedelta(days=d.date().weekday())
            for d in sorted_dates
        ))
        if not completion_weeks:
            return 0
        longest = 1
        current = 1
        for i in range(1, len(completion_weeks)):
            if (completion_weeks[i] - completion_weeks[i - 1]).days == 7:
                current += 1
                longest = max(longest, current)
            else:
                current = 1
        return longest

    def is_broken(self):
        if not self.completions:
            return False
        last_completion = max(self.completions)
        today = datetime.now()
        if self.periodicity == "daily":
            return (today - last_completion).days >= 1
        else:
            return (today - last_completion).days >= 7

    def get_summary(self):
        streak = self.get_streak()
        total_completions = len(self.completions)
        status = "🔴 BROKEN" if self.is_broken() else "🟢 Active"
        return (
            f"[{self.periodicity.upper()}] {self.name}\n"
            f"  Description : {self.description}\n"
            f"  Status      : {status}\n"
            f"  Streak      : {streak} {self.periodicity} periods\n"
            f"  Total done  : {total_completions} times"
        )

    def __str__(self):
        return self.get_summary()

## Step 3: Storage 

In [4]:
DATA_FILE = "habits.json"


def save(habits: list):
    """Save habits to JSON file."""
    data = []
    for habit in habits:
        habit_dict = {
            "name": habit.name,
            "description": habit.description,
            "periodicity": habit.periodicity,
            "created_date": habit.created_date.isoformat(),
            "completions": [c.isoformat() for c in habit.completions],
        }
        data.append(habit_dict)

    with open(DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"💾 Data saved to {DATA_FILE}")


def load():
    """Load habits from JSON file."""
    if not os.path.exists(DATA_FILE):
        print("📂 No data file found. Starting with empty habit list.")
        return []

    try:
        with open(DATA_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)

        habits = []
        for item in data:
            habit = Habit(
                name=item["name"],
                description=item["description"],
                periodicity=item["periodicity"],
            )
            habit.created_date = datetime.fromisoformat(item["created_date"])
            habit.completions = [
                datetime.fromisoformat(c) for c in item["completions"]
            ]
            habits.append(habit)

        print(f"📂 Loaded {len(habits)} habits from {DATA_FILE}")
        return habits

    except (json.JSONDecodeError, KeyError) as e:
        print(f"⚠️  Error reading {DATA_FILE}: {e}")
        print("Starting with empty habit list.")
        return []

## Step 4: Fixtures 

In [5]:
def create_fixtures():
    today = datetime.now()
    start = today - timedelta(days=28)

    # Habit 1: Read Market News (daily) - misses a handful of days
    habit1 = Habit(
        name="Read Market News",
        description="Read morning financial news before market opens",
        periodicity="daily",
    )
    habit1.created_date = start
    skip_days_habit1 = {5, 6, 13, 14, 19, 20, 21, 26, 27}
    for day_offset in range(28):
        if day_offset not in skip_days_habit1:
            completion_time = start + timedelta(days=day_offset, hours=8, minutes=15)
            habit1.completions.append(completion_time)

    # Habit 2: Check Portfolio (daily) - same missed days as habit 1
    habit2 = Habit(
        name="Check Portfolio",
        description="Review all open positions and their daily performance",
        periodicity="daily",
    )
    habit2.created_date = start
    skip_days_habit2 = {5, 6, 13, 14, 19, 20, 21, 26, 27}
    for day_offset in range(28):
        if day_offset not in skip_days_habit2:
            completion_time = start + timedelta(days=day_offset, hours=9, minutes=10)
            habit2.completions.append(completion_time)

    # Habit 3: Write Trading Journal (daily) - more gaps, broken streak
    habit3 = Habit(
        name="Write Trading Journal",
        description="Record every trade decision with its rationale and outcome",
        periodicity="daily",
    )
    habit3.created_date = start
    complete_days_habit3 = {0, 1, 2, 3, 4, 7, 8, 9, 10, 11, 12, 15, 16, 17, 18, 22, 23, 24, 25}
    for day_offset in range(28):
        if day_offset in complete_days_habit3:
            completion_time = start + timedelta(days=day_offset, hours=20, minutes=0)
            habit3.completions.append(completion_time)

    # Habit 4: Analyze Weekly Performance (weekly) - done 3 of 4 weeks
    habit4 = Habit(
        name="Analyze Weekly Performance",
        description="Full week review: P&L, what worked, and strategic adjustments",
        periodicity="weekly",
    )
    habit4.created_date = start
    weekly_completion_days_habit4 = {4, 11, 25}
    for day_offset in weekly_completion_days_habit4:
        completion_time = start + timedelta(days=day_offset, hours=18, minutes=30)
        habit4.completions.append(completion_time)

    # Habit 5: Review Risk Management Rules (weekly) - done every week
    habit5 = Habit(
        name="Review Risk Management Rules",
        description="Revisit position sizing and stop-loss rules to stay disciplined",
        periodicity="weekly",
    )
    habit5.created_date = start
    weekly_completion_days_habit5 = {3, 10, 17, 24}
    for day_offset in weekly_completion_days_habit5:
        completion_time = start + timedelta(days=day_offset, hours=17, minutes=0)
        habit5.completions.append(completion_time)

    return [habit1, habit2, habit3, habit4, habit5]

## Step 5: Analytics — functional-style helpers over a list of habits

In [6]:
def get_all_habits(habits: list):
    return list(habits)


def get_habits_by_period(habits: list, period: str):
    return list(filter(lambda h: h.periodicity == period, habits))


def get_longest_streak_all(habits: list):
    """Return the habit with the longest all-time streak."""
    if not habits:
        return None
    return max(habits, key=lambda h: h.get_longest_streak())


def get_longest_streak_for(habit):
    """Return the longest streak for one specific habit."""
    return habit.get_longest_streak()


def get_broken_habits(habits: list):
    return list(filter(lambda h: h.is_broken(), habits))


def get_habit_names(habits: list):
    return list(map(lambda h: h.name, habits))


def print_analytics_report(habits: list):
    """Print a full summary report of all habits."""
    if not habits:
        print("No habits found.")
        return

    print("\n📊 Habit Analytics Report")
    print(f"   Total habits: {len(habits)}")

    daily = get_habits_by_period(habits, "daily")
    weekly = get_habits_by_period(habits, "weekly")
    print(f"   Daily habits  : {len(daily)}")
    print(f"   Weekly habits : {len(weekly)}")

    broken = get_broken_habits(habits)
    if broken:
        print(f"\n⚠️  Broken habits ({len(broken)}):")
        for h in broken:
            print(f"   - {h.name}")
    else:
        print("\n✅ No broken habits — great work!")

    best = get_longest_streak_all(habits)
    if best:
        print(f"\n🏆 Longest streak overall: {best.name} ({best.get_longest_streak()} days)")
    else:
        print("\n🏆 No habits to analyze for longest streak.")

    print("\n📈 Current streaks:")
    for h in habits:
        streak = h.get_streak()
        bar = "█" * streak if streak > 0 else "—"
        print(f"   {h.name:<35} {streak:>3} {bar}")

    print("\n" + "=" * 55)

## Step 6: CLI — menu and screen

In [7]:
def show_menu():
    """Print the main menu to the screen."""
    print("\n" + "=" * 50)
    print("   📈  STOCK TRADING HABIT TRACKER")
    print("=" * 50)
    print("  [1]  Show all my habits")
    print("  [2]  Add a new habit")
    print("  [3]  Check off a habit (mark as done)")
    print("  [4]  Delete a habit")
    print("  [5]  View analytics report")
    print("  [6]  Load example trading habits (demo data)")
    print("  [0]  Exit")
    print("=" * 50)


def show_all_habits(habits: list):
    """Display all habits with their current status."""
    if not habits:
        print("\n⚠️  You have no habits yet. Add one or load demo data.")
        return

    print(f"\n📋 YOUR HABITS ({len(habits)} total):\n")
    for i, h in enumerate(habits, start=1):
        print(f"  {i}. {h}")
        print()

## Step 7: CLI — add, check off, delete

In [8]:
def add_habit(habits: list) -> list:
    print("\n── ADD NEW HABIT ─────────────────────────────")

    while True:
        name = input("  Habit name (e.g. 'Read Market News'): ").strip()
        if name:
            break
        print("  ⚠️  Name cannot be empty. Try again.")

    description = input("  Description: ").strip()
    if not description:
        description = "No description provided."

    while True:
        period = input("  Periodicity [daily / weekly]: ").strip().lower()
        if period in ("daily", "weekly"):
            break
        print("  ⚠️  Please type exactly 'daily' or 'weekly'.")

    new_habit = Habit(name=name, description=description, periodicity=period)
    habits.append(new_habit)
    save(habits)
    print(f"\n✅ Habit '{name}' added successfully!")
    return habits


def check_off_habit(habits: list) -> list:
    if not habits:
        print("\n⚠️  No habits to check off. Add some first.")
        return habits

    print("\n── CHECK OFF HABIT ───────────────────────────")
    for i, h in enumerate(habits, start=1):
        streak = h.get_streak()
        print(f"  [{i}] {h.name}  (current streak: {streak})")

    while True:
        try:
            choice = int(input("\n  Enter number (0 to cancel): "))
            if choice == 0:
                return habits
            if 1 <= choice <= len(habits):
                break
            print(f"  ⚠️  Please enter a number between 1 and {len(habits)}.")
        except ValueError:
            print("  ⚠️  Please enter a valid number.")

    selected = habits[choice - 1]
    selected.check_off()
    save(habits)
    print(f"  🔥 New streak: {selected.get_streak()} {selected.periodicity} periods!")
    return habits


def delete_habit(habits: list) -> list:
    if not habits:
        print("\n⚠️  No habits to delete.")
        return habits

    print("\n── DELETE HABIT ──────────────────────────────")
    for i, h in enumerate(habits, start=1):
        print(f"  [{i}] {h.name}")

    while True:
        try:
            choice = int(input("\n  Enter number to delete (0 to cancel): "))
            if choice == 0:
                return habits
            if 1 <= choice <= len(habits):
                break
            print(f"  ⚠️  Enter a number between 1 and {len(habits)}.")
        except ValueError:
            print("  ⚠️  Please enter a valid number.")
    selected = habits[choice - 1]
    confirm = input(f"\n  Delete '{selected.name}'? [yes / no]: ").strip().lower()
    if confirm == "yes":
        habits.pop(choice - 1)
        save(habits)
        print(f"  🗑️  '{selected.name}' deleted.")
    else:
        print("  ❌ Deletion cancelled.")
    return habits

def edit_habit(habits: list) -> list:
    if not habits:
        print("\n\u26a0\ufe0f  No habits to edit.")
        return habits
    print("\n\u2500\u2500 EDIT HABIT \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
    for i, h in enumerate(habits, start=1):
        print(f"  [{i}] {h.name}")
    selected = habits[choice - 1]
    print("  Leave a field blank to keep its current value.")
    new_name = input(f"  New name [{selected.name}]: ").strip()
    selected.edit(
        name=new_name or None,
    )
    save(habits)
    print(f"  \u2705 '{selected.name}' updated.")
    return habits

## Step 8: CLI — analytics view and demo data loader

In [9]:
def view_analytics(habits: list):
    """Show the full analytics report."""
    print_analytics_report(habits)

    if not habits:
        return

    print("\n── DETAILED ANALYTICS ────────────────────────")
    print("  [1] Show only daily habits")
    print("  [2] Show only weekly habits")
    print("  [3] Show habit with longest streak")
    print("  [0] Back to main menu")

    choice = input("\n  Choose: ").strip()

    if choice == "1":
        daily = get_habits_by_period(habits, "daily")
        print(f"\n📅 DAILY HABITS ({len(daily)}):")
        for h in daily:
            print(f"   - {h.name}  (streak: {h.get_streak()})")

    elif choice == "2":
        weekly = get_habits_by_period(habits, "weekly")
        print(f"\n📅 WEEKLY HABITS ({len(weekly)}):")
        for h in weekly:
            print(f"   - {h.name}  (streak: {h.get_streak()})")

    elif choice == "3":
        best = get_longest_streak_all(habits)
        if best:
            streak = get_longest_streak_for(best)
            print(f"\n🏆 Best habit: '{best.name}'")
            print(f"   Longest streak ever: {streak} {best.periodicity} periods")


def load_demo_data(habits: list) -> list:
    if habits:
        confirm = input(
            "\n  ⚠️  You already have habits. Replace all with demo data? [yes / no]: "
        ).strip().lower()
        if confirm != "yes":
            print("  ❌ Cancelled.")
            return habits

    habits = create_fixtures()
    save(habits)
    print("✅ Demo trading habits loaded and saved!")
    return habits

## Step 9: Main loop
Run this cell to start the app. Type `0` any time to exit the menu loop.

In [10]:
def main():
    """Main loop of the application."""
    print("\n👋 Welcome to the Stock Trading Habit Tracker!")

    habits = load()

    if not habits:
        answer = input("\n  No habits found. Load 5 demo trading habits? [yes / no]: ").strip().lower()
        if answer == "yes":
            habits = load_demo_data(habits)

    while True:
        show_menu()
        choice = input("  Your choice: ").strip()

        if choice == "1":
            show_all_habits(habits)
        elif choice == "2":
            habits = add_habit(habits)
        elif choice == "3":
            habits = check_off_habit(habits)
        elif choice == "4":
            habits = delete_habit(habits)
        elif choice == "5":
            view_analytics(habits)
        elif choice == "6":
            habits = load_demo_data(habits)
        elif choice == "0":
            print("\n👋 Goodbye! Keep tracking your trading habits!\n")
            break
        else:
            print("\n  ⚠️  Invalid choice. Please enter a number from the menu.")

In [11]:
habits = create_fixtures()
print_analytics_report(habits)
for h in habits:
    print(h)


📊 Habit Analytics Report
   Total habits: 5
   Daily habits  : 3
   Weekly habits : 2

⚠️  Broken habits (3):
   - Read Market News
   - Check Portfolio
   - Write Trading Journal

🏆 Longest streak overall: Read Market News (6 days)

📈 Current streaks:
   Read Market News                      0 —
   Check Portfolio                       0 —
   Write Trading Journal                 0 —
   Analyze Weekly Performance            1 █
   Review Risk Management Rules          4 ████

[DAILY] Read Market News
  Description : Read morning financial news before market opens
  Status      : 🔴 BROKEN
  Streak      : 0 daily periods
  Total done  : 19 times
[DAILY] Check Portfolio
  Description : Review all open positions and their daily performance
  Status      : 🔴 BROKEN
  Streak      : 0 daily periods
  Total done  : 19 times
[DAILY] Write Trading Journal
  Description : Record every trade decision with its rationale and outcome
  Status      : 🔴 BROKEN
  Streak      : 0 daily periods
  Total 

In [12]:
main()


👋 Welcome to the Stock Trading Habit Tracker!
📂 Loaded 2 habits from habits.json

   📈  STOCK TRADING HABIT TRACKER
  [1]  Show all my habits
  [2]  Add a new habit
  [3]  Check off a habit (mark as done)
  [4]  Delete a habit
  [5]  View analytics report
  [6]  Load example trading habits (demo data)
  [0]  Exit

  ⚠️  Invalid choice. Please enter a number from the menu.

   📈  STOCK TRADING HABIT TRACKER
  [1]  Show all my habits
  [2]  Add a new habit
  [3]  Check off a habit (mark as done)
  [4]  Delete a habit
  [5]  View analytics report
  [6]  Load example trading habits (demo data)
  [0]  Exit

  ⚠️  Invalid choice. Please enter a number from the menu.

   📈  STOCK TRADING HABIT TRACKER
  [1]  Show all my habits
  [2]  Add a new habit
  [3]  Check off a habit (mark as done)
  [4]  Delete a habit
  [5]  View analytics report
  [6]  Load example trading habits (demo data)
  [0]  Exit

📋 YOUR HABITS (2 total):

  1. [DAILY] 3
  Description : 5
  Status      : 🟢 Active
  Streak   